In [0]:
# --------------------------------------------------------------------------------------------------
# SCRIPT: 1.4_sobreposicao_ideologica_frentes
# LOCAL: 3_gold/src/1_atlas_frentes/
# OBJETIVO: Gerar tabelas com sobreposição de membros entre frentes ideologicamente opostas
#          (fonte 'gold_atlas_frentes_parlamentares' gerado pelo script 3_gold/1_atlas_frentes/1.1_atlas_frentes_consolidado) 
# ENTREGA: Sobreposição de membros entre frentes: quem está em frentes ideologicamente opostas?
# --------------------------------------------------------------------------------------------------

from pyspark.sql.functions import col, lower

# Carrega a Gold
df_atlas = spark.table("gold_atlas_frentes_parlamentares")

# Self-Join: Bancada do Soluções Digitais x Defesa das universidades públicas
df_agro = df_atlas.filter(lower(col("nome_frente")).contains("soluções digitais")).alias("sol_dig")
df_ambiental = df_atlas.filter(lower(col("nome_frente")).contains("defesa das universidades públicas")).alias("def_uni")

# Join pelo id_deputado para ver quem está em ambas
df_sobreposicao = df_agro.join(
    df_ambiental, 
    on="id_deputado", 
    how="inner"
).select(
    col("id_deputado"),
    col("sol_dig.nome_deputado").alias("parlamentar"), 
    col("sol_dig.partido").alias("partido"),
    col("sol_dig.nome_frente").alias("frente_A"),
    col("def_uni.nome_frente").alias("frente_B")
)

# Salvar resultado
df_sobreposicao.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_atlas_sobreposicao_sol_dig_def_uni")

display(df_sobreposicao)

In [0]:
# Deletar a tabela
spark.sql("DROP TABLE IF EXISTS gold_atlas_sobreposicao_ambiental_agropecuaria")

In [0]:
# Self-Join: Bancada do Nanismo x Bancada Ferroviária 
df_agro = df_atlas.filter(lower(col("nome_frente")).contains("nanismo")).alias("nanismo")
df_ambiental = df_atlas.filter(lower(col("nome_frente")).contains("ferroviária")).alias("ferrov")

# Join pelo id_deputado para ver quem está em ambas
df_sobreposicao_2 = df_agro.join(
    df_ambiental, 
    on="id_deputado", 
    how="inner"
).select(
    col("id_deputado"),
    col("nanismo.nome_deputado").alias("parlamentar"), 
    col("nanismo.partido").alias("partido"),
    col("nanismo.nome_frente").alias("frente_A"),
    col("ferrov.nome_frente").alias("frente_B")
)

# Salvar resultado
df_sobreposicao_2.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_atlas_sobreposicao_nanismo_ferroviaria")

display(df_sobreposicao_2)